In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

sys.path.append("..")

from config import *

In [2]:
feature_df = pd.read_csv("../DATA/CLEANED/transactions_cleaned_final.csv")

feature_df["timestamp"] = pd.to_datetime(feature_df["timestamp"])

users_df = pd.read_csv("../DATA/RAW/users_raw.csv")
merchants_df = pd.read_csv("../DATA/RAW/merchants_raw.csv")
user_device_mapping = pd.read_csv("../DATA/RAW/user_device_mapping_raw.csv")

print("Feature Engineering dataset loaded successfully.")
print("Shape:", feature_df.shape)

Feature Engineering dataset loaded successfully.
Shape: (200000, 17)


In [4]:
feature_df["is_late_night"] = (
    feature_df["timestamp"]
    .dt.hour
    .between(0,5)
)

In [5]:
feature_df["is_late_night"].value_counts()

is_late_night
False    194618
True       5382
Name: count, dtype: int64

In [7]:
# feature_df.shape

feature_df["is_late_night"].value_counts()

is_late_night
False    194618
True       5382
Name: count, dtype: int64

In [8]:
# Feature 2 – is_business_hours
feature_df["is_business_hours"] = (
    feature_df["timestamp"]
    .dt.hour
    .between(9,18)
)

In [9]:
feature_df["is_business_hours"].value_counts()

is_business_hours
True     110440
False     89560
Name: count, dtype: int64

In [10]:
# Feature 3 – rush_hour_flag
feature_df["rush_hour_flag"] = (
    feature_df["timestamp"]
    .dt.hour
    .isin([8,9,10,19,20,21])
)

In [11]:
feature_df["rush_hour_flag"].value_counts()

rush_hour_flag
False    109315
True      90685
Name: count, dtype: int64

In [12]:
conditions = [

    feature_df["is_late_night"],

    feature_df["rush_hour_flag"],

    feature_df["is_business_hours"]

]

choices = [

    3,

    1,

    0

]

feature_df["hour_risk_score"] = np.select(

    conditions,

    choices,

    default=2

)

In [18]:
# feature_df.shape

# feature_df["is_business_hours"].value_counts()

# feature_df["rush_hour_flag"].value_counts()

feature_df["hour_risk_score"].value_counts().sort_index()

hour_risk_score
0    81311
1    90685
2    22622
3     5382
Name: count, dtype: int64

In [19]:
# Feature 5 – device_usage_count
device_usage = (
    user_device_mapping
    .groupby("device_id")["user_id"]
    .nunique()
    .reset_index(name="device_usage_count")
)

device_usage.head()

,device_id,device_usage_count
0,DEV00001,1
1,DEV00003,1
2,DEV00004,1
3,DEV00005,1
4,DEV00006,1


In [20]:
feature_df = feature_df.merge(
    device_usage,
    on="device_id",
    how="left"
)

In [21]:
feature_df["device_usage_count"].describe()

count    200000.000000
mean          1.237255
std           0.587781
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           5.000000
Name: device_usage_count, dtype: float64

In [22]:
# Feature 6 – shared_device_flag
feature_df["shared_device_flag"] = (
    feature_df["device_usage_count"] > 1
)

In [25]:
feature_df["shared_device_flag"].value_counts()

shared_device_flag
False    162294
True      37706
Name: count, dtype: int64

In [30]:
# feature_df.shape

# feature_df["device_usage_count"].describe()

# feature_df["shared_device_flag"].value_counts()

feature_df[["device_id","device_usage_count","shared_device_flag"]].head(10)

,device_id,device_usage_count,shared_device_flag
0,DEV06706,1,False
1,DEV08663,1,False
2,DEV11055,1,False
3,DEV03254,1,False
4,DEV10449,1,False
5,DEV06223,1,False
6,DEV09520,1,False
7,DEV01192,1,False
8,DEV02530,2,True
9,DEV03300,1,False


In [31]:
user_activity = (
    feature_df
    .groupby("user_id")
    .size()
    .reset_index(name="total_transactions")
)

user_activity.head()

,user_id,total_transactions
0,U00001,14
1,U00002,30
2,U00003,14
3,U00004,19
4,U00005,13


In [32]:
user_activity["user_transaction_rank"] = (
    user_activity["total_transactions"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

In [33]:
feature_df = feature_df.merge(
    user_activity[["user_id","total_transactions","user_transaction_rank"]],
    on="user_id",
    how="left"
)

In [34]:
feature_df[[
    "user_id",
    "total_transactions",
    "user_transaction_rank"
]].head()

,user_id,total_transactions,user_transaction_rank
0,U03855,23,57
1,U03984,10,70
2,U03867,18,62
3,U04995,13,67
4,U08774,14,66


In [35]:
feature_df["transaction_date"] = (
    feature_df["timestamp"].dt.date
)

In [36]:
daily_user_activity = (
    feature_df
    .groupby(["user_id","transaction_date"])
    .size()
    .reset_index(name="daily_transaction_count")
)

In [37]:
feature_df = feature_df.merge(
    daily_user_activity,
    on=["user_id","transaction_date"],
    how="left"
)

In [38]:
feature_df["burst_activity_flag"] = (
    feature_df["daily_transaction_count"] >= 5
)

In [44]:
# feature_df.shape

# feature_df[["user_id","total_transactions","user_transaction_rank"]].head(10)

# feature_df["user_transaction_rank"].describe()

# feature_df["daily_transaction_count"].describe()

# feature_df["burst_activity_flag"].value_counts()

feature_df[["user_id","transaction_date","daily_transaction_count","burst_activity_flag"]].head(10)

,user_id,transaction_date,daily_transaction_count,burst_activity_flag
0,U03855,2026-01-22,1,False
1,U03984,2026-04-02,1,False
2,U03867,2026-03-18,1,False
3,U04995,2026-03-18,1,False
4,U08774,2026-02-09,1,False
5,U00862,2026-02-23,1,False
6,U06416,2026-02-18,1,False
7,U04543,2026-05-17,1,False
8,U04395,2026-01-11,1,False
9,U03064,2026-02-10,1,False


In [45]:
feature_df["burst_activity_flag"] = (
    feature_df["daily_transaction_count"] >= 4
)

feature_df["burst_activity_flag"].value_counts()

burst_activity_flag
False    199852
True        148
Name: count, dtype: int64

In [46]:
# Feature 9 – merchant_risk_score
feature_df = feature_df.merge(
    merchants_df[["merchant_id", "merchant_category"]],
    on="merchant_id",
    how="left"
)

In [47]:
feature_df["merchant_category"].isnull().sum()

np.int64(0)

In [48]:
merchant_risk = (
    feature_df
    .groupby("merchant_category")["suspicion_flag"]
    .mean()
    .mul(100)
    .reset_index(name="merchant_risk_score")
)

In [49]:
feature_df = feature_df.merge(
    merchant_risk,
    on="merchant_category",
    how="left"
)

In [50]:
feature_df[["merchant_category", "merchant_risk_score"]].head()

,merchant_category,merchant_risk_score
0,Utilities,5.499782
1,Grocery,5.681163
2,Grocery,5.681163
3,Food,5.553104
4,Food,5.553104


In [52]:
# Feature 10 – user_activity_score
conditions = [
    feature_df["total_transactions"] >= 30,
    feature_df["total_transactions"].between(20, 29),
    feature_df["total_transactions"].between(10, 19)
]

choices = [3, 2, 1]

feature_df["user_activity_score"] = np.select(
    conditions,
    choices,
    default=0
)

In [53]:
feature_df["user_activity_score"].value_counts().sort_index()

user_activity_score
0     5052
1    87845
2    43058
3    64045
Name: count, dtype: int64

In [54]:
# Feature 11 – overall_risk_score ⭐ (Project USP)
feature_df["overall_risk_score"] = 0

In [55]:
# Late Night
feature_df.loc[
    feature_df["is_late_night"],
    "overall_risk_score"
] += 3

# Shared Device
feature_df.loc[
    feature_df["shared_device_flag"],
    "overall_risk_score"
] += 2

# Failed Payment
feature_df.loc[
    feature_df["payment_status"] == "Failed",
    "overall_risk_score"
] += 2

# High Value
feature_df.loc[
    feature_df["amount"] >= 50000,
    "overall_risk_score"
] += 3

# Burst Activity
feature_df.loc[
    feature_df["burst_activity_flag"],
    "overall_risk_score"
] += 2

# User Activity
feature_df["overall_risk_score"] += feature_df["user_activity_score"]

# Merchant Risk Bonus
feature_df.loc[
    feature_df["merchant_risk_score"] >= 5.8,
    "overall_risk_score"
] += 1

In [56]:
feature_df["risk_category"] = pd.cut(
    feature_df["overall_risk_score"],
    bins=[-1, 2, 5, 20],
    labels=["Low", "Medium", "High"]
)

In [64]:
# feature_df.shape

# feature_df["burst_activity_flag"].value_counts()

# feature_df["user_activity_score"].value_counts().sort_index()

# feature_df["overall_risk_score"].describe()

# feature_df["risk_category"].value_counts()

feature_df[["overall_risk_score","risk_category"]].head(10)

,overall_risk_score,risk_category
0,2,Low
1,1,Low
2,1,Low
3,1,Low
4,1,Low
5,2,Low
6,2,Low
7,1,Low
8,3,Medium
9,3,Medium


In [65]:
# 🚀 Feature 12 – investigation_priority
feature_df["investigation_priority"] = np.select(
    [
        feature_df["risk_category"] == "High",
        feature_df["risk_category"] == "Medium"
    ],
    [
        "P1",
        "P2"
    ],
    default="P3"
)

In [66]:
feature_df["investigation_priority"].value_counts()

investigation_priority
P3    102770
P2     93208
P1      4022
Name: count, dtype: int64

In [78]:
# 🚀 Feature 13 – risk_reason (Explainable Fraud Detection)
def generate_risk_reason(row):

    reasons = []

    if row["is_late_night"]:
        reasons.append("Late Night")

    if row["shared_device_flag"]:
        reasons.append("Shared Device")

    if row["payment_status"] == "Failed":
        reasons.append("Failed Payment")

    if row["amount"] >= 50000:
        reasons.append("High Value")

    if row["burst_activity_flag"]:
        reasons.append("Burst Activity")

    if row["merchant_risk_score"] >= 5.8:
        reasons.append("High Risk Merchant")
    if row["user_activity_score"] >= 2:
        reasons.append("High Activity User")    

    if not reasons:
        return "Normal"

    return " | ".join(reasons)

In [79]:
feature_df["risk_reason"] = feature_df.apply(
    generate_risk_reason,
    axis=1
)

In [80]:
feature_df["risk_reason"].value_counts().head(10)

risk_reason
High Activity User                         79702
Normal                                     68616
Shared Device | High Activity User         18211
Shared Device                              16264
Failed Payment | High Activity User         3462
Failed Payment                              3090
Late Night | High Activity User             2169
Late Night                                  1955
High Risk Merchant | High Activity User     1081
High Risk Merchant                           959
Name: count, dtype: int64

In [70]:
feature_df[
    ["overall_risk_score","risk_category","risk_reason"]
].sample(10)

,overall_risk_score,risk_category,risk_reason
119737,5,Medium,Shared Device
72272,1,Low,Normal
158154,2,Low,Normal
65426,2,Low,Normal
30074,1,Low,Normal
23677,1,Low,Normal
134858,3,Medium,Normal
176418,3,Medium,Normal
132467,3,Medium,Normal
4082,2,Low,Normal


In [82]:
feature_df.to_csv(
    "../DATA/CLEANED/transactions_feature_engineered.csv",
    index=False
)

In [83]:
pd.read_csv(
    "../DATA/CLEANED/transactions_feature_engineered.csv"
).shape

(200000, 35)

In [77]:
# feature_df.shape

# feature_df["investigation_priority"].value_counts()

# feature_df["risk_reason"].value_counts().head(10)

# feature_df[["overall_risk_score","risk_category","investigation_priority","risk_reason"]].sample(10)

pd.read_csv("../DATA/CLEANED/transactions_feature_engineered.csv").shape

(200000, 35)

In [81]:
feature_df[
    (feature_df["overall_risk_score"]>=3)
    &
    (feature_df["risk_reason"]=="Normal")
].shape

(0, 35)